# Getting All Data

In [1]:
from tikapi import TikAPI, ValidationException, ResponseException
import pandas as pd
import json
import re
import os
import requests
import datetime
from requests.exceptions import RequestException
from retrying import retry
import numpy as np
import matplotlib.pyplot as plt

In [2]:
#API Keys
api = TikAPI('FvjyMY0ZTgyUAKBF2WThiNoNKIdUl0QqYv1v3NtKHRB53Iwe')
User =  api.user('c_M13O1BVCA8')

In [3]:
directories = [
    'alphapose_json/apple',
    'alphapose_json/savage',
    'alphapose_json/sayso',
    'alphapose_json/cannibal',
    'alphapose_json/supalonely',
    #'alphapose_json/nopole'
]

all_files = []
dance_names = []
#file_lim = 200 #file limit used to overcome API limit

for directory in directories:
    files = os.listdir(directory)#[file_lim*4:file_lim*5]
    dance_name = directory.split('/')[-1]
    
    dance_names.extend([dance_name] * len(files)) 
    all_files.extend(files)

tiktok_data = pd.DataFrame({
    'alphapose_file': all_files,
    'Dance': dance_names
})

In [4]:
tiktok_data.shape

(5000, 2)

In [5]:
#load json files
def load_json(row):
    dance_folder = row['Dance'] #folder name (same as dance)
    filename = row['alphapose_file'] #filename
    json_path = os.path.join("alphapose_json", dance_folder, filename) #get path
    
    if os.path.exists(json_path):
        with open(json_path, 'r') as file:
            return json.load(file) #load file
    return None

In [6]:
tiktok_data["alphapose_coordinates"] = tiktok_data.apply(load_json, axis=1)

**Step 3 - normalisation**

In [7]:
#extract coordinates and normalise them 
def normalise_coordinates(row):
    alphapose = row['alphapose_coordinates']
        
    frames = {}
    for frame in alphapose:
        image_id = frame.get('image_id').split('.')[0] #get the frame number
        keypoints = frame.get('keypoints') #get keypoints
        
        if image_id not in frames: #one set of coordinates per frame (ensures only 1 person per video)
            frames[image_id] = {'keypoints': keypoints}
    
    normalised_frames = []
    for image_id, data in sorted(frames.items()):
        keypoints = data['keypoints']
        frame_coords = []
        
        for i in range(0, len(keypoints), 3):
            x, y, score = keypoints[i:i+3]
            #take those with confidence scores above 0.5 otherwise (0,0)
            if score >= 0.5:
                frame_coords.append((x, y))
            else:
                frame_coords.append((0, 0))
        
        #normalise by nose
        nose_x, nose_y = frame_coords[0]
        normalised_frame = []
        for x, y in frame_coords:
            normalised_frame.extend([x - nose_x, y - nose_y])
        normalised_frames.append(normalised_frame)
        
    return np.array(normalised_frames)

In [ ]:
tiktok_data["normalised_alphapose"] = tiktok_data.apply(normalise_coordinates, axis=1)

In [9]:
#check for empty alphapose
empty_rows = tiktok_data[tiktok_data['alphapose_coordinates'].apply(lambda x: isinstance(x, list) and len(x) == 0)]

empty_files_info = empty_rows[['alphapose_file', 'Dance']]

print("Files with empty alphapose coordinates:")
print(empty_files_info)


Files with empty alphapose coordinates:
Empty DataFrame
Columns: [alphapose_file, Dance]
Index: []


**Step 4 - dance classifier**

In [10]:
#apply dance classifier to videos
def dance_classification(model, row):
    alphapose = row['normalised_alphapose']
    predictions = model.predict(alphapose) #predict dancing based on coords
    if np.mean(predictions) >= 0.5:
        return 'yes'
    else:
        return 'no'

In [11]:
import xgboost as xgb

dance_classifier = xgb.XGBClassifier() #import best performing classifier
dance_classifier.load_model("XGBoost_dance_classifier.json")

In [12]:
#apply classifiers
tiktok_data['dancing'] = tiktok_data.apply(lambda row: dance_classification(dance_classifier, row), axis = 1)

In [13]:
tiktok_data['dancing'].value_counts()

dancing
no     3501
yes    1499
Name: count, dtype: int64

**Step 5 - video metrics**

In [14]:
#calculate people in each video
def calculate_people(row):
    people = 0
    alphapose = row['alphapose_coordinates']
    for frame in alphapose:
        image_id = frame.get('image_id').split('.')[0]
        if image_id == '60': #60 frames accounts for a few seconds into the video
            people += 1 #counts how many times the frame is in the json
    return people
        
    

In [15]:
tiktok_data["people"] = tiktok_data.apply(calculate_people, axis=1)

In [16]:
#extract the video id from the filename
def extract_video_id(row):
    filename = row['alphapose_file']
    video_id = filename.split('-')[2].split('_')[1].split('.')[0]
    return video_id

In [17]:
tiktok_data['video_id'] = tiktok_data.apply(extract_video_id, axis = 1)

In [18]:
tiktok_data

,alphapose_file,Dance,alphapose_coordinates,normalised_alphapose,dancing,people,video_id
0,alphapose-results-video_7401324200356531461.json,apple,"[{'image_id': '0.jpg', 'category_id': 1, 'keyp...","[[0.0, 0.0, 10.020523071289062, -3.34017944335...",no,4,7401324200356531461
1,alphapose-results-video_7423107618173947141.json,apple,"[{'image_id': '0.jpg', 'category_id': 1, 'keyp...","[[0.0, 0.0, 18.106781005859375, -25.3494873046...",no,2,7423107618173947141
2,alphapose-results-video_7405720331949853957.json,apple,"[{'image_id': '0.jpg', 'category_id': 1, 'keyp...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...",no,0,7405720331949853957
3,alphapose-results-video_7397542554331303174.json,apple,"[{'image_id': '0.jpg', 'category_id': 1, 'keyp...","[[0.0, 0.0, 21.1041259765625, -14.069427490234...",yes,1,7397542554331303174
4,alphapose-results-video_7391225934390545694.json,apple,"[{'image_id': '0.jpg', 'category_id': 1, 'keyp...","[[0.0, 0.0, 44.438995361328125, -35.5512084960...",yes,1,7391225934390545694
...,...,...,...,...,...,...,...
4995,alphapose-results-video_6816419042933427461.json,supalonely,"[{'image_id': '0.jpg', 'category_id': 1, 'keyp...","[[0.0, 0.0, 349.3280029296875, 285.69497680664...",no,2,6816419042933427461
4996,alphapose-results-video_6824816249600625925.json,supalonely,"[{'image_id': '259.jpg', 'category_id': 1, 'ke...","[[0.0, 0.0, -361.29046630859375, -222.61585998...",no,0,6824816249600625925
4997,alphapose-results-video_6805285237585726725.json,supalonely,"[{'image_id': '0.jpg', 'category_id': 1, 'keyp...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 315.4113769531...",no,1,6805285237585726725
4998,alphapose-results-video_6797776399668088070.json,supalonely,"[{'image_id': '0.jpg', 'category_id': 1, 'keyp...","[[0.0, 0.0, 12.93817138671875, -12.93820190429...",yes,6,6797776399668088070


**Step 5**

In [19]:
#get comments, likes, account follower counts and description from each video
def fetch_video_information(api, row):
    video_id = row['video_id']
    info = {'followers': 'unavailable', 'likes': 'unavailable', 'comments': 'unavailable', 'description': 'unavailable'}
    print(f"Processing video: {video_id}")
    
    try:
        video_response = api.public.video(id=video_id)
        video_json = video_response.json()
        
        #extract author stats
        item_info = video_json.get('itemInfo', {})
        item_struct = item_info.get('itemStruct', {})
        author_stats = item_struct.get('authorStats', {})
        info['followers'] = author_stats.get('followerCount', 'unavailable')

        #extract description and engagement data
        share_meta = video_json.get('shareMeta', {})
        desc_text = share_meta.get('desc', '')

        #extract likes and comments correctly as strings
        likes_match = re.search(r'([\d\.]+[KM]?) likes', desc_text)
        comments_match = re.search(r'([\d\.]+[KM]?) comments', desc_text)
        description_match = re.search(r'comments\. (.+)$', desc_text)

        #store in dictionary
        info['likes'] = f"{likes_match.group(1)} likes" if likes_match else 'unavailable'
        info['comments'] = f"{comments_match.group(1)} comments" if comments_match else 'unavailable'
        info['description'] = description_match.group(1) if description_match else 'unavailable'

        return info

    except ValidationException as e:
        print(e, e.field)
    except ResponseException as e:
        print(e, e.response.status_code)
    except KeyError as e:
        print(f"Missing key: {e} in video ID {video_id}")
    except Exception as e:
        print(f"Unexpected error: {e}")


In [20]:
#check on one video, also see rough time for one video
import time

start = time.time()
x = fetch_video_information(api, tiktok_data.iloc[1])
print(x)
end = time.time()

print('total time: ', end - start)

Processing video: 7423107618173947141
{'followers': 7000000, 'likes': '88.4K likes', 'comments': '1080 comments', 'description': '“#duet with @Brianna #POV You live in a world where you can only be positive or else (slime asmr version)..”'}
total time:  11.023520708084106


In [30]:
#apply to all videos
import time

start = time.time()

tiktok_data["video_metrics"] = tiktok_data.apply(lambda row: fetch_video_information(api, row), axis=1)

end = time.time()

print('total time: ', end - start)

Processing video: 7401324200356531461


KeyboardInterrupt: 

In [ ]:
#change the filname based on what dance
#tiktok_data.to_csv('nopole5.csv', index = False)

In [31]:
nopole_csv_files = [
    "nopole1.csv",
    "nopole2.csv",
    "nopole3.csv",
    "nopole4.csv",
    "nopole5.csv"
]

#read and combine all csv files into a single df
all_nopole = [pd.read_csv(file) for file in nopole_csv_files]
nopole_all = pd.concat(all_nopole, ignore_index=True)

In [ ]:
#save all videos for each soundtrack(dance trend)
#nopole_all.to_csv('nopole_all.csv', index = False)

In [32]:
all_csv_files = [
    "nopole_all.csv",
    "all_videos.csv"
]

# Read and combine all CSV files into a single df
all_vids = [pd.read_csv(file) for file in all_csv_files]
all_videos_2 = pd.concat(all_vids, ignore_index=True)

In [ ]:
#all_videos_2.to_csv('all_videos_2.csv', index = False)